# 数学作为推理的试金石：AlphaGeometry 与 AlphaProof


> 上一讲我们沿着推理模型的思路，研究了怎样让 LLM 在生成答案时多花一步思考：思维链、自洽性、测试时计算。这些方法让推理变长，却没有正面回答一个更根本的问题——当任务要求绝对正确、半点含糊都不行时，模型的推理需要达到怎样的可靠程度。
>
> 这一讲把这个问题放到最苛刻的场景：国际数学奥林匹克（IMO）。我们沿着三条技术路线走一遍——AlphaGeometry 用神经-符号结合解几何题，AlphaProof 把证明变成强化学习可搜索的游戏，Gemini Deep Think 用纯自然语言在比赛时限内拿到官方认证的金牌。每一路我们都亲手实现其中的最小组件：一个迷你几何符号引擎、一个逐行检查的证明验证器，以及一个生成候选、验证、推进的搜索循环。

数学竞赛题成为推理试金石的原因有三条。可验证：一道题的对错有客观判据，证明过程要么成立要么不成立，不能靠看起来合理蒙混过关。奖励稀疏：一道 IMO 题满分 7 分，但没有渐进分——证出来就是 7 分，差一步就是 0 分，中间没有任何信号告诉模型它接近了。需要搜索与工具：竞赛题的证明往往有几十步，单次生成几乎必然在某一步卡住，必须在巨大的证明空间里搜索，还要能提出新的辅助构造、借助形式化验证器把关。

这三条合起来，让数学与开放对话形成鲜明对比。对话任务里，一段错误但流畅的回答常被当作大致正确；数学任务里，这种含糊会被一个绝对公正、永不犯错的裁判当场判零分。本讲先看这条判分规则带来的后果，再从几何到数论，逐一搭建求解它的组件。

## 1. 数学：推理的试金石

先看一个失败案例。下面这段证明声称 2 等于 1，是经典的除以零骗局：设 a = b，两边同乘 a 得 a² = ab，两边减 b² 得 a² − b² = ab − b²，即 (a+b)(a−b) = b(a−b)。两边同除以 (a−b) 得 a+b = b，代回 a = b 得 2b = b，于是 2 = 1。

如果这段写在考卷上，人一眼能看出问题：因为 a = b，所以 a − b = 0，除以 0 是非法的。难处在于一个逐行处理符号的机器，若不检查除以 0 这条规则，也会一路算到底。自然语言证明的每一步都没有被机器验证，错误藏在某一行里，任何看起来合理的中间结果都不能保证结论正确。这正是数学任务与对话任务的分野：前者需要一个逐行把关的裁判，后者没有。


In [ ]:
import numpy as np
np.random.seed(42)

# 手算：IMO 单题判分——要么满分要么零分，没有中间档位
def imo_score(proved):
    """单题判分：形式化证出得满分 7 分，否则 0 分。"""
    return 7 if proved else 0

cases = [
    ("完整证出并验证", True),
    ("差最后一步辅助构造", False),
    ("思路正确但有一行非法", False),
    ("完全没做", False),
]
for name, ok in cases:
    print(f"{name:<12} -> {imo_score(ok)} 分")

# 手算：证明搜索空间的规模
b = 20          # 每个状态下平均候选动作数
d = 50          # 典型竞赛证明的步数
print(f"分支因子 b={b}，证明深度 d={d}，搜索树叶节点约 {b ** d:.1e}")
print("关键观察：奖励只有 0 或 7 两种，而搜索空间是 10 的几十次方量级")


In [ ]:
import matplotlib.pyplot as plt

depths = np.array([5, 10, 20, 30, 50])
leaves = 20.0 ** depths

plt.figure(figsize=(6, 4))
plt.semilogy(depths, leaves, marker="o")
plt.xlabel("proof depth d")
plt.ylabel("search tree leaves (log scale)")
plt.title("Sparse reward meets a huge search space")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


## 2. AlphaGeometry：几何的神经-符号结合

AlphaGeometry 拆开几何证明的两种能力。符号引擎 DDAR 负责确定性推理，由两部分组成：Deductive Database（DD）是一组 Horn 子句式演绎规则，比如等腰三角形两底角相等、全等三角形对应边相等；Algebraic Reasoning（AR）解角度、比例与距离的方程。DD 的规则形如：条件是 isos(A,B,C)，结论是角 abc 等于角 acb。这类由条件合取推出一个结论的规则每条都确定、可验证，引擎反复套用到不再产生新事实为止。

DDAR 有一个短板：需要辅助构造的题会彻底卡住。辅助构造指题面之外新加的点或线，比如取 BC 的中点 M、过 A 作 BC 的垂线，等价于在搜索中生成一个外部项。AlphaGeometry 用一个 150M 参数的 decoder-only transformer 在引擎卡住时提议下一步加什么辅助点，然后交给 DDAR 继续推。

我们把这条分工搬进一个最小版本。下面的迷你几何符号引擎只处理一个 toy 定理：已知三角形 ABC 等腰（AB = AC，顶点 A），且 M 是底边 BC 的中点，求证角 AMB 等于角 AMC。规则库如下：

- R1 `isos(A,B,C)` → `ang_eq(ABC, ACB)`：等腰三角形两底角相等
- R2 `isos(A,B,C)` → `seg_eq(AB, AC)`：等腰定义，两腰相等
- R3 `mid(M,B,C)` → `seg_eq(BM, CM)`：中点定义，BM = CM
- R4 `mid(M,B,C)` ∧ `isos(A,B,C)` → `ang_eq(ABC, ABM)`：M 在 BC 上，两角是同一个角
- R5 `mid(M,B,C)` ∧ `isos(A,B,C)` → `ang_eq(ACB, ACM)`：同理，C 端共线同角
- R6 `ang_eq(x,y)` ∧ `ang_eq(y,z)` → `ang_eq(x,z)`：角相等传递
- R7 `ang_eq(x,y)` → `ang_eq(y,x)`：角相等对称
- R8 边角边：`seg_eq(AB,AC)` ∧ `seg_eq(BM,CM)` ∧ `ang_eq(ABM,ACM)` → `congr(ABM,ACM)`
- R9 `congr(ABM,ACM)` → `ang_eq(AMB,AMC)`：全等三角形对应角相等

In [ ]:
# 手算：不加辅助点的版本 A 只有 isos(A,B,C)，推两步就停
base_steps = [
    ("R1_底角相等", ("ang_eq", "A", "B", "C", "A", "C", "B")),
    ("R2_等腰定义", ("seg_eq", "A", "B", "A", "C")),
]
print("版本 A（无辅助点）可做的演绎：")
for rule, concl in base_steps:
    print(f"  由 isos(A,B,C) 推出 {concl}   依据 {rule}")

# 手算：加入辅助构造 mid(M,B,C) 后，演绎链条可以继续
print("\n版本 B（加入辅助点 M 与中点关系）的完整手算链条：")
hand_steps = [
    ("R1_底角相等", ("ang_eq", "A", "B", "C", "A", "C", "B")),
    ("R2_等腰定义", ("seg_eq", "A", "B", "A", "C")),
    ("R3_中点定义", ("seg_eq", "B", "M", "C", "M")),
    ("R4_共线同角左", ("ang_eq", "A", "B", "C", "A", "B", "M")),
    ("R5_共线同角右", ("ang_eq", "A", "C", "B", "A", "C", "M")),
    ("R6_角相等传递", ("ang_eq", "A", "B", "C", "A", "C", "M")),
    ("R7_角相等对称", ("ang_eq", "A", "B", "M", "A", "B", "C")),
    ("R6_角相等传递", ("ang_eq", "A", "B", "M", "A", "C", "M")),
    ("R8_边角边", ("congr", "A", "B", "M", "A", "C", "M")),
    ("R9_全等对应角", ("ang_eq", "A", "M", "B", "A", "M", "C")),
]
for rule, concl in hand_steps:
    print(f"  推出 {concl}   依据 {rule}")
goal = ("ang_eq", "A", "M", "B", "A", "M", "C")
print("目标", goal, "在版本 B 中可达，在版本 A 中不可达")


In [ ]:
# 从零实现：迷你几何符号引擎（Horn 子句式前向演绎）
def match(pattern, fact, env):
    """把模式与事实匹配，变量形如 '?X'，返回绑定字典或 None。"""
    if env is None:
        return None
    if isinstance(pattern, str) and pattern.startswith("?"):
        if pattern in env:
            return env if env[pattern] == fact else None
        env = dict(env)
        env[pattern] = fact
        return env
    if isinstance(pattern, tuple):
        if not isinstance(fact, tuple) or len(pattern) != len(fact):
            return None
        for p, f in zip(pattern, fact):
            env = match(p, f, env)
            if env is None:
                return None
        return env
    return env if pattern == fact else None

def instantiate(pattern, env):
    """用绑定字典把模式里的变量替换为具体值。"""
    if isinstance(pattern, str):
        return env.get(pattern, pattern)
    if isinstance(pattern, tuple):
        return tuple(instantiate(p, env) for p in pattern)
    return pattern

def enumerate_matches(antecedents, facts, idx, env):
    """枚举让所有前提在事实库上成立的环境，供前向链接使用。"""
    if idx == len(antecedents):
        yield env
        return
    for fact in facts:
        new_env = match(antecedents[idx], fact, env)
        if new_env is not None:
            yield from enumerate_matches(antecedents, facts, idx + 1, new_env)

class GeoEngine:
    """迷你几何符号引擎：反复应用 Horn 规则直到不再产生新事实。"""

    def __init__(self, rules):
        self.rules = rules  # 每条规则是 (名字, 前提模式列表, 结论模式)

    def forward_chain(self, premises, max_rounds=300):
        """前向演绎：从前提出发推出全部可推事实，返回 (事实列表, 追踪记录)。"""
        facts = list(premises)
        trace = []
        for _ in range(max_rounds):
            added = False
            for name, antecedents, conclusion in self.rules:
                for env in enumerate_matches(antecedents, facts, 0, {}):
                    new = instantiate(conclusion, env)
                    if new not in facts:
                        facts.append(new)
                        trace.append((name, new))
                        added = True
                        break
            if not added:
                break
        return facts, trace

# 冒烟测试：模式匹配与实例化
print("示例匹配：",
      match(("mid", "?M", "?B", "?C"), ("mid", "M", "B", "C"), {}))
print("示例实例化：",
      instantiate(("seg_eq", "?B", "?M"), {"?B": "B", "?M": "M"}))


In [ ]:
RULES = [
    ("R1_底角相等", [("isos", "?A", "?B", "?C")],
     ("ang_eq", "?A", "?B", "?C", "?A", "?C", "?B")),
    ("R2_等腰定义", [("isos", "?A", "?B", "?C")],
     ("seg_eq", "?A", "?B", "?A", "?C")),
    ("R3_中点定义", [("mid", "?M", "?B", "?C")],
     ("seg_eq", "?B", "?M", "?C", "?M")),
    ("R4_共线同角左",
     [("mid", "?M", "?B", "?C"), ("isos", "?A", "?B", "?C")],
     ("ang_eq", "?A", "?B", "?C", "?A", "?B", "?M")),
    ("R5_共线同角右",
     [("mid", "?M", "?B", "?C"), ("isos", "?A", "?B", "?C")],
     ("ang_eq", "?A", "?C", "?B", "?A", "?C", "?M")),
    ("R6_角相等传递",
     [("ang_eq", "?A", "?B", "?C", "?X", "?Y", "?Z"),
      ("ang_eq", "?X", "?Y", "?Z", "?P", "?Q", "?R")],
     ("ang_eq", "?A", "?B", "?C", "?P", "?Q", "?R")),
    ("R7_角相等对称", [("ang_eq", "?A", "?B", "?C", "?X", "?Y", "?Z")],
     ("ang_eq", "?X", "?Y", "?Z", "?A", "?B", "?C")),
    ("R8_边角边",
     [("seg_eq", "?A", "?B", "?A", "?C"),
      ("seg_eq", "?B", "?M", "?C", "?M"),
      ("ang_eq", "?A", "?B", "?M", "?A", "?C", "?M")],
     ("congr", "?A", "?B", "?M", "?A", "?C", "?M")),
    ("R9_全等对应角", [("congr", "?A", "?B", "?M", "?A", "?C", "?M")],
     ("ang_eq", "?A", "?M", "?B", "?A", "?M", "?C")),
]

engine = GeoEngine(RULES)
goal = ("ang_eq", "A", "M", "B", "A", "M", "C")
facts, trace = engine.forward_chain([
    ("isos", "A", "B", "C"),
    ("mid", "M", "B", "C"),
])
print("引擎推出的全部事实：")
for f in facts:
    print("  ", f)
print("目标", goal, "被推出：", goal in facts)
print("追踪记录步数：", len(trace))


上面的引擎把推出新事实自动化了。现在回到辅助构造。把前提去掉 mid(M,B,C)，只留 isos(A,B,C)，引擎能推出的只有底角相等、两腰相等以及它们的对称形式，然后停住——M 在知识库里不存在，任何涉及 M 的规则都触发不了，目标角 AMB 与 AMC 无从谈起。这就是 DDAR 的卡住状态。

AlphaGeometry 的处理是：引擎卡住时，让语言模型提议一个辅助构造，把新事实加进前提，引擎继续推。这条分工是本讲第一个可复用模式——神经网络提议候选（辅助点、辅助线），符号引擎确定性验证并推进。注意验证由引擎保证：LM 只是提出加一个中点 M 并声明 BM = CM，之后每一条演绎都由 Horn 规则逐一确认，LM 的输出不会跳过任何一步。

In [ ]:
# 实验：不加辅助点 vs 加辅助构造，对比能推出的结论集合
goal = ("ang_eq", "A", "M", "B", "A", "M", "C")

facts_a, trace_a = engine.forward_chain([("isos", "A", "B", "C")])
print("版本 A（无辅助点）推出", len(facts_a), "条事实：")
for f in facts_a:
    print("  ", f)
print("目标是否推出：", goal in facts_a)

facts_b, trace_b = engine.forward_chain([
    ("isos", "A", "B", "C"),
    ("mid", "M", "B", "C"),          # 辅助构造：取底边 BC 的中点 M
])
print("\n版本 B（加辅助点 M 与中点关系）推出", len(facts_b), "条事实：")
for f in facts_b:
    print("  ", f)
print("目标是否推出：", goal in facts_b)
print("新增事实数：", len(facts_b) - len(facts_a))


In [ ]:
# 可视化：神经-符号分工——语言模型提议辅助构造，符号引擎确定性演绎
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axis("off")

def box(x, y, w, h, text, color):
    p = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02",
                       facecolor=color, edgecolor="black", linewidth=1.2)
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=10)

box(0.02, 0.52, 0.30, 0.30, "symbolic engine\nDD: horn rules\n(stuck)", "#dbe9f7")
box(0.38, 0.52, 0.30, 0.30, "LLM\npropose auxiliary\nmid(M, B, C)", "#e7f2d8")
box(0.74, 0.52, 0.24, 0.30, "engine\nresumes\n(deduce)", "#dbe9f7")
box(0.34, 0.06, 0.34, 0.22, "verified proof\nang_eq(AMB, AMC)", "#fbe6d5")

ax.annotate("", xy=(0.38, 0.67), xytext=(0.32, 0.67),
            arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(0.74, 0.67), xytext=(0.68, 0.67),
            arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(0.51, 0.28), xytext=(0.86, 0.52),
            arrowprops=dict(arrowstyle="->", lw=1.5))
ax.text(0.33, 0.72, "propose aux", fontsize=9)
ax.text(0.69, 0.72, "feed facts", fontsize=9)
ax.text(0.72, 0.34, "deduce", fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.title("Neuro-symbolic division of labor in AlphaGeometry")
plt.tight_layout()
plt.show()


## 3. AlphaProof：形式化证明的 RL

AlphaGeometry 的符号引擎是硬编码的领域规则，只覆盖几何。AlphaProof 换了一个更通用的底子：形式语言 Lean。Lean 能把任意数学命题与证明变成机器可逐行检查的对象——推理的每一步都是一条合法变换，验证器绝对可靠，不会放过任何非法步骤。把数学证明当成游戏：当前证明状态是局面，每一步动作改写状态，Lean 检查器是游戏规则。AlphaZero 式的强化学习就在这个游戏里搜索并自我对弈。

人类手写的形式化证明太少，这是第一道瓶颈。AlphaProof 用微调过的 Gemini 当形式化器，把自然语言题目自动翻译成形式化陈述，造出百万量级的不同难度题库，再让求解器在难题上反复自我强化。我们无力复现这套大规模训练，但可以复现它依赖的最小核心：一个逐行检查的证明验证器，以及一个生成候选、验证、推进的搜索循环。

先实现验证器。下面的迷你理论处理偶数：谓词 even(n) 表示存在整数 k 使得 n = 2k。要证的是：若 even(a) 且 even(b)，则 even(a+b)。证明对象是一系列行，每行是（语句、规则、依赖行号），验证器逐行确认每一条都合法。手算一遍：

1. even(a)  前提
2. even(b)  前提
3. a = 2k  由 1 展开偶数的定义（k 是新见证变量）
4. b = 2l  由 2 展开定义（l 是新见证变量）
5. a + b = 2k + 2l  由 3、4 代入加法
6. 2k + 2l = 2(k + l)  分配律（归一化下成立）
7. a + b = 2(k + l)  由 5、6 传递
8. even(a + b)  由 7 与见证 k + l 收回定义

In [ ]:
# 手算验证一步：归一化下 2k + 2l 与 2(k + l) 恒等
def var(name):
    """构造变量项。"""
    return ("var", name)

def add(t1, t2):
    """构造加法项。"""
    return ("add", t1, t2)

def mul(c, t):
    """构造常量倍项，c 是整数。"""
    return ("mul", c, t)

def normalize(t):
    """把项规约到规范形式，用于判定代数式恒等。"""
    if t[0] == "var":
        return t
    if t[0] == "add":
        parts = []
        for sub in t[1:]:
            n = normalize(sub)
            if n[0] == "add":
                parts.extend(n[1:])
            else:
                parts.append(n)
        return ("add",) + tuple(sorted(parts))
    if t[0] == "mul":
        c, inner = t[1], normalize(t[2])
        if inner[0] == "add":
            subs = [normalize(("mul", c, s)) for s in inner[1:]]
            return normalize(("add",) + tuple(subs))
        if c == 1:
            return inner
        return ("mul", c, inner)
    return t

k, l = var("k"), var("l")
left = add(mul(2, k), mul(2, l))          # 2k + 2l
right = mul(2, add(k, l))                 # 2(k + l)
print("left  =", normalize(left))
print("right =", normalize(right))
print("归一化下恒等：", normalize(left) == normalize(right))


In [ ]:
# 从零实现：行式证明验证器——逐行检查规则、依赖与归一化等值
def vars_of(t):
    """收集项或语句中出现的全部变量名。"""
    if isinstance(t, tuple):
        if t[0] == "var":
            return {t[1]}
        out = set()
        for sub in t[1:]:
            out |= vars_of(sub)
        return out
    return set()

def is_fresh(proof, i, premises, name):
    """见证变量 name 在第 i 行之前是否完全未出现过。"""
    seen = set()
    for stmt in premises:
        seen |= vars_of(stmt)
    for j in range(i):
        seen |= vars_of(proof[j][1])
    return name not in seen

def check_line(proof, i, premises):
    """检查第 i 行是否由依赖行按规则合法推出，返回 (是否合法, 原因)。"""
    rule, stmt, deps = proof[i]
    if rule == "premise":
        return stmt in premises, "必须是给定前提"
    if rule == "arith":
        return (stmt[0] == "eq" and
                normalize(stmt[1]) == normalize(stmt[2])), "归一化后等值"
    if rule == "witness":
        if len(deps) != 1:
            return False, "见证需要恰好一个依赖行"
        dep_stmt = proof[deps[0]][1]
        t = stmt[1]
        if dep_stmt[0] != "even" or normalize(dep_stmt[1]) != normalize(t):
            return False, "依赖行必须是 even(t)"
        if stmt[0] != "eq" or stmt[2][0] != "mul" or stmt[2][1] != 2:
            return False, "语句必须是 t = 2k"
        w = stmt[2][2]
        if w[0] != "var" or not is_fresh(proof, i, premises, w[1]):
            return False, "见证变量必须全新"
        return True, "见证展开"
    if rule == "compose":
        if len(deps) != 2:
            return False, "加法代入需要两个依赖行"
        d1, d2 = (proof[j][1] for j in deps)
        if d1[0] != "eq" or d2[0] != "eq":
            return False, "依赖行必须是等式"
        if stmt[0] != "eq":
            return False, "语句必须是等式"
        if (normalize(stmt[1]) != normalize(("add", d1[1], d2[1]))
                or normalize(stmt[2]) != normalize(("add", d1[2], d2[2]))):
            return False, "语句与代和不符"
        return True, "加法代入"
    if rule == "trans":
        if len(deps) != 2:
            return False, "传递需要两个依赖行"
        d1, d2 = (proof[j][1] for j in deps)
        if d1[0] != "eq" or d2[0] != "eq":
            return False, "依赖行必须是等式"
        if (normalize(d1[2]) != normalize(d2[1])
                or normalize(stmt[1]) != normalize(d1[1])
                or normalize(stmt[2]) != normalize(d2[2])):
            return False, "中间项衔接不上"
        return True, "等式传递"
    if rule == "even_def":
        if len(deps) != 1:
            return False, "收回定义需要恰好一个依赖行"
        dep_stmt = proof[deps[0]][1]
        if (dep_stmt[0] != "eq" or dep_stmt[2][0] != "mul"
                or dep_stmt[2][1] != 2):
            return False, "依赖行必须是 t = 2s"
        if stmt[0] != "even" or normalize(stmt[1]) != normalize(dep_stmt[1]):
            return False, "语句必须是 even(t)"
        return True, "收回定义"
    return False, "未知规则"

def verify(proof, premises):
    """逐行检查整条证明，返回 (是否全部合法, 每行报告)。"""
    ok_all = True
    report = []
    for i in range(len(proof)):
        ok, reason = check_line(proof, i, premises)
        report.append((i, proof[i][0], ok, reason))
        if not ok:
            ok_all = False
    return ok_all, report

# 冒烟测试：加法交换律在归一化下自动成立
probe = [("arith", ("eq", add(var("a"), var("b")),
                    add(var("b"), var("a"))), [])]
print("冒烟测试：", check_line(probe, 0, []))


In [ ]:
# 构造 even(a) ∧ even(b) → even(a+b) 的行式证明并运行验证器
a, b = var("a"), var("b")
k, l = var("k"), var("l")
premises = [("even", a), ("even", b)]

proof = [
    ("premise",  ("even", a),                                     []),
    ("premise",  ("even", b),                                     []),
    ("witness",  ("eq", a, ("mul", 2, k)),                        [0]),
    ("witness",  ("eq", b, ("mul", 2, l)),                        [1]),
    ("compose",  ("eq", ("add", a, b),
                  ("add", ("mul", 2, k), ("mul", 2, l))),         [2, 3]),
    ("arith",    ("eq", ("add", ("mul", 2, k), ("mul", 2, l)),
                  ("mul", 2, ("add", k, l))),                     []),
    ("trans",    ("eq", ("add", a, b),
                  ("mul", 2, ("add", k, l))),                     [4, 5]),
    ("even_def", ("even", ("add", a, b)),                         [6]),
]

ok_all, report = verify(proof, premises)
for i, rule, ok, reason in report:
    print(f"行 {i}: 规则 {rule:<8} 合法={ok}  ({reason})")
print("整条证明合法：", ok_all)


In [ ]:
# 篡改演示：把第 5 行的结论改成 a + b = 2k，验证器必须当场判负
bad = [list(line) for line in proof]
bad[5][1] = ("eq", ("add", a, b), ("mul", 2, k))
ok_bad, report_bad = verify(bad, premises)
print("篡改后的证明是否通过：", ok_bad)
for i, rule, ok, reason in report_bad:
    if not ok:
        print(f"  非法行 {i}: 规则 {rule} 被否决（{reason}）")


验证器证明了一件关键的事：证明的每一步都能被机器逐行检查。现在把镜头拉回 AlphaProof。形式化之后，数学证明成了一个可以搜索的游戏：状态是当前到达的结论，动作是从一条规则实例出下一步，验证器是游戏规则。AlphaProof 用强化学习搜索这个游戏，我们用更朴素的方式做同样的事——实现一个通用的搜索循环：从当前状态生成候选步骤，验证器把关，通过的推进、失败的换一个候选，直到推出目标或穷尽。

下面在一个人工的蕴涵图上演示这个抽象框架。六个命题 p0 到 p5，规则 r1 到 r6 表示命题间的蕴含，其中 r3 通向死路 p4，r6 通向目标 p5。从初始结论 p0 出发，搜索的每一步都会被验证器逐条确认。


In [ ]:
# 从零实现：搜索 + 验证器的抽象循环
SEARCH_RULES = {
    "r1": ("p0", "p1"),
    "r2": ("p0", "p2"),
    "r3": ("p1", "p4"),   # 死路：p4 不再有任何规则可用
    "r4": ("p1", "p3"),
    "r5": ("p2", "p3"),
    "r6": ("p3", "p5"),   # 通往目标
}

def applicable(state, rules):
    """生成器：返回从当前结论出发所有可应用的规则实例。"""
    cands = []
    for name, (premise, conclusion) in rules.items():
        if premise == state:
            cands.append((name, conclusion))
    return cands

def verify_step(state, conclusion, rule, rules):
    """验证器：规则前提是当前状态、结论与规则一致才算合法。"""
    if rule not in rules:
        return False, "未知规则"
    premise, expected = rules[rule]
    if premise != state:
        return False, "前提不是当前状态"
    if conclusion != expected:
        return False, "结论与规则不符"
    return True, ""

def search_dfs(start, goal, rules, max_depth=8):
    """深度优先搜索一条证明路径，死路回退，返回 (路径, 访问节点数)。"""
    stack = [(start, [])]
    visited = {start}
    nodes = 0
    while stack:
        state, path = stack.pop()
        nodes += 1
        if state == goal:
            return path, nodes
        if len(path) >= max_depth:
            continue
        for name, conclusion in reversed(applicable(state, rules)):
            if conclusion not in visited:
                visited.add(conclusion)
                stack.append((conclusion, path + [(name, conclusion)]))
    return None, nodes

def check_path(path, rules, start, goal):
    """把一条证明路径逐条喂给验证器，返回 (是否全部合法, 是否到目标)。"""
    state = start
    for name, conclusion in path:
        ok, _ = verify_step(state, conclusion, name, rules)
        if not ok:
            return False, state == goal
        state = conclusion
    return True, state == goal

# 冒烟测试：生成器在当前状态下能给出哪些候选
print("从 p0 出发的可应用候选：", applicable("p0", SEARCH_RULES))


In [ ]:
# 运行搜索：从 p0 出发找一条到 p5 的证明路径
path, nodes = search_dfs("p0", "p5", SEARCH_RULES)
print("搜索访问的节点数：", nodes)
for step, (name, conclusion) in enumerate(path):
    print(f"  第 {step + 1} 步：应用 {name} -> 推出 {conclusion}")

legal, reached = check_path(path, SEARCH_RULES, "p0", "p5")
print("路径逐条验证合法：", legal)
print("最终到达目标：", reached)
print("死路 p4 被访问后回退，未出现在最终路径里：",
      "p4" not in [c for _, c in path])


In [ ]:
# 可视化：蕴涵图上的搜索，绿色是找到的路径，红色是死路
pos = {
    "p0": (0.0, 0.0),
    "p1": (-1.0, -1.0),
    "p2": (1.0, -1.0),
    "p4": (-1.9, -2.0),
    "p3": (0.0, -2.0),
    "p5": (0.5, -3.0),
}
edges = [
    ("r1", "p0", "p1", "path"), ("r2", "p0", "p2", "gray"),
    ("r3", "p1", "p4", "dead"), ("r4", "p1", "p3", "path"),
    ("r5", "p2", "p3", "gray"), ("r6", "p3", "p5", "path"),
]

fig, ax = plt.subplots(figsize=(6, 4.5))
for name, u, v, kind in edges:
    (x1, y1), (x2, y2) = pos[u], pos[v]
    color = {"gray": "#b0b0b0", "dead": "#d64541", "path": "#2e8b57"}[kind]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.8))
    ax.text((x1 + x2) / 2 + 0.08, (y1 + y2) / 2, name, fontsize=9, color=color)
for node, (x, y) in pos.items():
    is_dead = node == "p4"
    ax.plot(x, y, "o", markersize=11, color="#d64541" if is_dead else "#4c9be8")
    ax.text(x + 0.1, y + 0.06, node, fontsize=10)
ax.text(-1.0, -2.7, "p4: dead end (backtrack)", fontsize=8, color="#d64541")
ax.text(0.05, -3.5, "p5: goal reached", fontsize=8, color="#2e8b57")
ax.set_xlim(-2.5, 2.2)
ax.set_ylim(-3.7, 0.4)
ax.axis("off")
plt.title("DFS on the implication graph (green = found path)")
plt.tight_layout()
plt.show()


## 4. IMO 金牌：Gemini 的路线

AlphaProof 的路线把可靠验证放在第一位：题目先翻译成 Lean，证明每一步被机器确认，代价是需要形式化翻译与动辄数天的算力。Gemini Deep Think 走了另一条路——完全放弃形式化，直接读官方题面，用自然语言写出严谨证明，在比赛时限内完成。2025 年的 IMO 上它以 35/42 分（6 题做对 5 题）达到金牌线，成绩由 IMO 协调员按与人类选手完全相同的标准官方认证。这一年它反而超过了自己的前身：2024 年 AlphaProof 与 AlphaGeometry 2 合起来是 28/42，还要两三天算力。

两条路线的取舍值得说清楚。形式验证路线追求绝对可靠：验证器不放任何非法步骤过关，但速度慢、成本高，且要把题先形式化。自然语言 RL 路线追求又快又通用：不需要翻译，模型直接读题，但证明对不对的验证是开放问题，最终靠人评分。AlphaProof 的证明搜索里，模型提出候选步骤、形式化验证器把关，这个神经-符号分工两条路线都保留——差别只在把关者是人还是机器。

下面把上一节的搜索框架接上 llm_client：让模型提出下一步证明步骤，形式化验证器逐条把关，循环直到证明完成或失败。mock 模式下模型的输出为脚本化占位，演示代码全部可离线运行。


In [ ]:
# 统一 LLM 客户端：有 API key 用真实模型，否则自动进入 mock 模式
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()

# 模型输出的约定：STEP: <结论> RULE: <规则名>，验证器据此逐行把关
import re

def parse_steps(text):
    """从模型输出中解析 (结论, 规则名) 列表，格式 STEP: X RULE: r。"""
    return re.findall(r"STEP:\s*(\S+)\s*RULE:\s*(\S+)", text)

# mock 模式下用脚本化输出演示解析与验证路径（真实 API 输出完整推理）
scripted = (
    "STEP: p1 RULE: r1\n"
    "STEP: p2 RULE: r5\n"      # 非法：r5 需要前提 p2，当前状态是 p1
    "STEP: p3 RULE: r4\n"
    "STEP: p5 RULE: r6\n"
)
steps = parse_steps(scripted)
print("mock 模式输出为占位，解析出的步骤数：", len(steps))

state = "p0"
report = []
for idx, (conclusion, rule) in enumerate(steps):
    ok, reason = verify_step(state, conclusion, rule, SEARCH_RULES)
    report.append((idx, conclusion, rule, ok, reason))
    if ok:
        state = conclusion
for idx, conclusion, rule, ok, reason in report:
    print(f"步骤 {idx}: {conclusion} / {rule}  合法={ok}  ({reason})")
failed = [idx for idx, _, _, ok, _ in report if not ok]
print("被判非法的步骤号：", failed)


In [ ]:
def propose_next(state, goal, rules, client):
    """让模型提议下一步证明步骤。mock 模式下用一步前瞻避开死路。"""
    if client.is_mock:
        for name, (premise, conclusion) in rules.items():
            if premise != state:
                continue
            if conclusion == goal or applicable(conclusion, rules):
                return [(conclusion, name)]
        for name, (premise, conclusion) in rules.items():
            if premise == state:
                return [(conclusion, name)]
        return []
    prompt = (
        "当前状态: " + state + "\n"
        "目标: " + goal + "\n"
        "规则库: " + ", ".join(
            f"{n}: {p} -> {c}" for n, (p, c) in rules.items()) + "\n"
        "请输出下一步，格式 STEP: <结论> RULE: <规则名>"
    )
    reply = client.chat([{"role": "user", "content": prompt}], temperature=0.2)
    return parse_steps(reply)

def run_proof_loop(start, goal, rules, client, max_rounds=10):
    """LLM 提议下一步，验证器把关，循环到证明完成或轮数耗尽。"""
    state = start
    log = []
    for rnd in range(max_rounds):
        if state == goal:
            return log, True
        for conclusion, rule in propose_next(state, goal, rules, client):
            ok, reason = verify_step(state, conclusion, rule, rules)
            if ok:
                state = conclusion
                log.append((rnd, conclusion, rule, "accepted"))
                break
            log.append((rnd, conclusion, rule, "rejected: " + reason))
    return log, state == goal

log, done = run_proof_loop("p0", "p5", SEARCH_RULES, client)
for rnd, conclusion, rule, status in log:
    print(f"轮 {rnd}: 候选 {conclusion} / {rule} -> {status}")
print("证明完成：", done)


In [ ]:
# 可视化：两条路线的对比——形式验证 vs 自然语言 RL
routes = ["formal Lean\nIMO 2024\n(~2-3 days)",
          "natural language\nIMO 2025\n(within 4.5 h)"]
scores = [28, 35]

fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(routes, scores, color=["#9bb7d4", "#7ed6a5"])
ax.axhline(29, color="#d64541", linestyle="--", linewidth=1.2)
ax.text(0.9, 30, "gold threshold 29/42", color="#d64541", fontsize=9)
for b, s in zip(bars, scores):
    ax.text(b.get_x() + b.get_width() / 2, s + 0.5, str(s),
            ha="center", fontsize=11)
ax.set_ylim(0, 42)
ax.set_ylabel("score out of 42")
ax.set_title("Two routes to IMO gold")
plt.tight_layout()
plt.show()


**两条路线的合流**

把三篇论文串起来，就是一条通往超级人类推理的演进线：纯符号引擎只能解固定的领域，加一个神经提议器就有了辅助构造的能力（AlphaGeometry）；把符号引擎换成形式化证明系统，再配上强化学习搜索，就能覆盖任意数学命题（AlphaProof）；当模型与强化学习足够强，形式化这层保险甚至可以省掉，直接读题、写自然语言证明，在时限内拿到金牌（Gemini）。对 Agent 课程而言，数学的意义在于它为任何想在推理上变强的 Agent 提供了一块可验证的训练与评测场地。下一讲我们把这些组件拼成一个自治智能体，可靠性问题会再次出现。

## 小结

- [ ] 数学是推理的试金石：可验证、奖励稀疏、需要搜索与工具，三条合起来让含糊论证无处遁形
- [ ] 迷你几何符号引擎：Horn 子句式规则反复前向演绎直到闭包，DD 与 AR 是 DDAR 的两半
- [ ] 辅助构造解锁推导：不加辅助点引擎卡住，加入中点 M 后边角边全等链条推到目标
- [ ] 神经-符号分工：语言模型提议候选步骤，符号引擎确定性验证并推进
- [ ] 最小证明验证器：行式证明逐条检查规则、依赖与归一化等值，非法步骤被当场标出
- [ ] 搜索 + 验证器框架：生成候选、验证把关、死路回退，AlphaProof 证明搜索的骨架
- [ ] 两条路线对照：形式验证可靠但慢，自然语言 RL 快而通用，2025 年 Gemini 在时限内拿下官方金牌

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

三道小题都基于本讲写过的组件，先动手补全，再运行断言验证。


**作业 1：验证器逐行把关**

补全 check_line，判断证明中第 i 行是否由依赖行按规则合法推出。证明采用（语句、规则名、依赖行号）的行式格式，规则库 RULES 给出每条规则的前提与结论。

小提示：先取第 i 行的规则名，再核对它的前提列表是否恰好等于依赖行的语句、结论是否等于本行语句；premise 规则直接放行。


In [ ]:
# 作业 1：补全 check_line，让合法证明全部通过、篡改行被判非法
HW1_RULES = {
    "r1": (["p0"], "p1"),
    "r2": (["p0"], "p2"),
    "r3": (["p1"], "p4"),
    "r4": (["p1"], "p3"),
    "r6": (["p3"], "p5"),
}

hw1_proof = [
    ("p0", "premise", []),
    ("p1", "r1", [0]),
    ("p3", "r4", [1]),
    ("p5", "r6", [2]),
]

def check_line(proof, i, rules):
    """判断第 i 行是否由依赖行按规则合法推出，返回 True/False。"""
    stmt, rule, deps = proof[i]
    if rule == "premise":
        return True
    premises, conclusion = rules[rule]
    # 填空：依赖行的语句恰好等于规则前提，且本行语句等于规则结论
    dep_stmts = [proof[j][0] for j in deps]
    return dep_stmts == premises and stmt == conclusion

assert all(check_line(hw1_proof, i, HW1_RULES) for i in range(len(hw1_proof)))
bad = list(hw1_proof)
bad[2] = ("p3", "r4", [0])      # 把依赖从第 1 行改成第 0 行
assert not check_line(bad, 2, HW1_RULES)
print("学会：验证器逐行检查，依赖不对应规则前提的行会被判非法")


**作业 2：搜索一条证明路径**

补全 find_proof，在蕴涵图上做深度优先搜索，返回一条从初始结论到目标的证明路径与访问节点数。规则库包含一条死路 r3，搜索必须回退才能到达目标。

小提示：维护当前状态与已访问集合，每步收集所有前提是当前状态、结论尚未访问的规则实例；超过 max_depth 就剪枝。


In [ ]:
# 作业 2：补全 find_proof，用 DFS 找一条合法证明路径
HW2_RULES = {
    "r1": ("p0", "p1"),
    "r2": ("p0", "p2"),
    "r3": ("p1", "p4"),
    "r4": ("p1", "p3"),
    "r5": ("p2", "p3"),
    "r6": ("p3", "p5"),
}

def find_proof(start, goal, rules, max_depth=8):
    """深度优先搜索证明路径，返回 (路径, 访问节点数)，找不到时路径为 None。"""
    stack = [(start, [])]
    visited = {start}
    nodes = 0
    while stack:
        state, path = stack.pop()
        nodes += 1
        if state == goal:
            return path, nodes
        if len(path) >= max_depth:
            continue
        candidates = []
        for name, (premise, conclusion) in rules.items():
            # 填空：前提是当前状态、结论尚未访问时才能入选
            if premise == state and conclusion not in visited:
                candidates.append((name, conclusion))
        for name, conclusion in reversed(candidates):
            visited.add(conclusion)
            stack.append((conclusion, path + [(name, conclusion)]))
    return None, nodes

path, nodes = find_proof("p0", "p5", HW2_RULES)
assert path is not None
state = "p0"
for name, conclusion in path:
    premise, expected = HW2_RULES[name]
    assert premise == state
    assert conclusion == expected
    state = conclusion
assert state == "p5"
print("学会：DFS 找到", len(path), "步证明，访问", nodes, "个节点，路径为", path)


**作业 3：解析模型输出并让验证器把关**

补全 parse_steps，从带 STEP/RULE 标记的文本中解析出 (结论, 规则名) 列表，再逐条用 verify_line 检查。文本里混入了一条非法步骤，验证器必须把它标出来。

小提示：用 re.findall 匹配 STEP: <结论> RULE: <规则名> 的模式，变量字符用 \S+；再顺序验证，只有通过的行才更新当前状态。


In [ ]:
# 作业 3：补全 parse_steps，解析模型输出并逐条验证
import re

HW3_RULES = {
    "r1": ("p0", "p1"),
    "r2": ("p0", "p2"),
    "r4": ("p1", "p3"),
    "r6": ("p3", "p5"),
}

hw3_text = (
    "STEP: p1 RULE: r1\n"
    "STEP: p3 RULE: r4\n"
    "STEP: p5 RULE: r6\n"
    "STEP: p2 RULE: r2\n"      # 非法：r2 需要前提 p0，当前状态是 p5
)

def parse_steps(text):
    """从 STEP/RULE 标记中解析出 (结论, 规则名) 列表。"""
    # 填空：按出现顺序匹配标记对
    return re.findall(r"STEP:\s*(\S+)\s*RULE:\s*(\S+)", text)

def verify_line(state, conclusion, rule, rules):
    """验证一步：规则存在、前提是当前状态。返回 (是否合法, 原因)。"""
    if rule not in rules:
        return False, "未知规则"
    premise, expected = rules[rule]
    if premise != state:
        return False, "前提不是当前状态"
    if conclusion != expected:
        return False, "结论与规则不符"
    return True, ""

steps = parse_steps(hw3_text)
state = "p0"
report = []
for idx, (conclusion, rule) in enumerate(steps):
    ok, reason = verify_line(state, conclusion, rule, HW3_RULES)
    report.append((idx, conclusion, rule, ok, reason))
    if ok:
        state = conclusion

assert len(steps) == 4
failed = [idx for idx, _, _, ok, _ in report if not ok]
assert failed == [3]
for idx, conclusion, rule, ok, reason in report:
    print(f"步骤 {idx}: {conclusion} / {rule}  合法={ok}  ({reason})")
print("学会：解析出的步骤全部喂给验证器，非法步骤被标出", failed)


## 参考资料

- Trinh et al., [AlphaGeometry: An Olympiad-level AI system for geometry](https://www.nature.com/articles/s41586-023-06747-5), Nature 2024, DOI:10.1038/s41586-023-06747-5 — 神经-符号几何证明系统：语言模型提议辅助构造，符号引擎 DDAR 确定性演绎，IMO-AG-30 上 25/30
- [AI achieves silver-medal standard solving International Mathematical Olympiad problems](https://deepmind.google/discover/blog/ai-solves-imo-problems-at-silver-medal-level/), DeepMind 博客 2024-07-25 — AlphaProof 与 AlphaGeometry 2 在 IMO 2024 得 28/42 银牌成绩的官方公告
- [AlphaProof technical report](https://www.nature.com/articles/s41586-025-09833-y), Nature 2025, DOI:10.1038/s41586-025-09833-y — 证明网络、AND-OR 树搜索、自动形式化管线与 TTRL 的完整技术细节
- [Advanced version of Gemini with Deep Think officially achieves gold-medal standard at the IMO](https://deepmind.google/blog/advanced-version-of-gemini-with-deep-think-officially-achieves-gold-medal-standard-at-the-international-mathematical-olympiad/), DeepMind 博客 2025-07 — Gemini Deep Think 在 IMO 2025 以 35/42 拿到官方认证金牌
- [The Lean Theorem Prover](https://lean-lang.org/) — AlphaProof 依赖的形式证明助手与依赖类型语言
- [miniF2F](https://github.com/openai/miniF2F) — 488 道竞赛级定理的形式与非形式配对基准，评估定理证明器与自动形式化器
